In [0]:
%run "/Users/patelrahul2614@gmail.com/databrick_demo/Digital_Banking_LakeHouse_Capstone/includes"

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Read the Bronze customers table
bronze_customers_df = spark.read.table(f"{catalog}.bronze.customers")
bronze_customers_df_dropped = bronze_customers_df.drop(col("file_name"),col("file_path"),col("ingestion_date"))
# using the orderby to get the updated customers
bronze_customers_df_desc = bronze_customers_df_dropped.orderBy(desc("updated_at"))

# 1. Remove duplicate customer records
deduped_df = bronze_customers_df_desc.dropDuplicates()

# 2. Trim leading and trailing spaces on all string columns
string_cols = [f.name for f in deduped_df.schema.fields if f.dataType.simpleString() == "string"]
trimmed_df = deduped_df
for c in string_cols:
    trimmed_df = trimmed_df.withColumn(c, trim(col(c)))

# 3. Validate email addresses — flag invalid values
email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
if "email" in string_cols:
    trimmed_df = trimmed_df.withColumn(
        "email_valid",
        when(col("email").isNull() | (col("email") == ""), lit(False))
        .otherwise(col("email").rlike(email_pattern)),
    )
    # Replace invalid emails with NULL so downstream consumers can handle them
    trimmed_df = trimmed_df.withColumn(
        "email", when(col("email_valid"), col("email")).otherwise(lit(None).cast("string"))
    )

# 4. Validate dates — flag rows where a date column cannot be parsed
date_cols = [
    c for c in trimmed_df.columns
    if c.lower() in ("dob", "date_of_birth", "created_at", "updated_at", "registration_date", "signup_date", "date")
]
for c in date_cols:
    if c in trimmed_df.columns:
        parsed = to_date(col(c))
        trimmed_df = trimmed_df.withColumn(
            f"{c}_valid", when(col(c).isNull(), lit(None)).otherwise(parsed.isNotNull())
        )
        # Replace unparseable date strings with NULL
        trimmed_df = trimmed_df.withColumn(
            c, when(col(f"{c}_valid"), parsed).otherwise(lit(None))
        )

# 5. Identify invalid records (invalid email or invalid date)
invalid_conditions = lit(False)
if "email" in string_cols and "email_valid" in trimmed_df.columns:
    invalid_conditions = invalid_conditions | (~col("email_valid"))
for c in date_cols:
    flag_col = f"{c}_valid"
    if flag_col in trimmed_df.columns:
        invalid_conditions = invalid_conditions | (col(flag_col) == lit(False))

dropped_df = trimmed_df.filter(invalid_conditions)
cleaned_df = trimmed_df.filter(~invalid_conditions)

# 6. Drop quality flag columns before writing
cleaned_df = cleaned_df.drop("email_valid", *[f"{c}_valid" for c in date_cols])
dropped_df = dropped_df.drop("email_valid", *[f"{c}_valid" for c in date_cols])

# 7. Write cleaned records to silver_cleaned table
cleaned_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.silver_customers")

# 8. Write invalid records to dropped_customers table
dropped_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.dropped_customers")